In [ ]:
# Cell 1: Imports + Classes
import os, gc, warnings
import numpy as np
import pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import joblib, shap
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Dropout, Input, BatchNormalization, ReLU, LeakyReLU
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.utils import to_categorical
from tensorflow.keras import regularizers
from tensorflow.keras.optimizers import Adam
from scikeras.wrappers import KerasClassifier
from sklearn.metrics import confusion_matrix, classification_report, f1_score
from sklearn.preprocessing import LabelEncoder, label_binarize
from sklearn.metrics import precision_recall_curve, average_precision_score

warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
np.random.seed(0); tf.random.set_seed(0)

OUT = 'pr_shap_out_Purified'
os.makedirs(OUT, exist_ok=True)

# Must be defined before joblib.load so pickle can resolve the class
class HistoryLogger(tf.keras.callbacks.Callback):
    def on_train_end(self, logs=None):
        self.model.final_history = self.model.history.history

def get_keras_model(m):
    km = getattr(m, 'model_', None)
    if km is None:
        try:
            km = m.named_steps['nn'].model_
        except Exception:
            pass
    return km

COLORS = plt.cm.tab20.colors
print('Setup done.')


In [2]:
# Cell 2: Config
# (display_label, pkl_filename, dataset_key, is_ae)
# dataset_key: 'tcga' | 'norway' | 'tcga_purified' | 'norway_purified'
MODELS = [
    # ── Non-Purified  TCGA ────────────────────────────────────────────────────
    ('NN-RS',                'NN_RS_bestmodel.pkl',                   'tcga',            False),
    ('NN-RS-AE',             'NN_RS_AE_bestmodel.pkl',                'tcga',            True ),
    ('NN-IRUS-RS',           'NN_IRUS_RS_bestmodel.pkl',              'tcga',            False),
    ('NN-IRUS-RS-AE',        'NN_IRUS_RS_AE_bestmodel.pkl',           'tcga',            True ),
    ('NN-SMOTE-RS',          'NN_SMOTE_RS_bestmodel.pkl',             'tcga',            False),
    ('NN-SMOTE-RS-AE',       'NN_SMOTE_RS_AE_bestmodel.pkl',          'tcga',            True ),
    # ── Purified  TCGA ────────────────────────────────────────────────────────
    ('NN-RS [P]',            'NN_RS_Purified_bestmodel.pkl',            'tcga_purified',   False),
    ('NN-RS-AE [P]',         'NN_RS_AE_Purified_bestmodel.pkl',         'tcga_purified',   True ),
    ('NN-IRUS-RS [P]',       'NN_IRUS_RS_Purified_bestmodel.pkl',       'tcga_purified',   False),
    ('NN-IRUS-RS-AE [P]',    'NN_IRUS_RS_AE_Purified_bestmodel.pkl',    'tcga_purified',   True ),
    ('NN-SMOTE-RS [P]',      'NN_SMOTE_RS_Purified_bestmodel.pkl',      'tcga_purified',   False),
    ('NN-SMOTE-RS-AE [P]',   'NN_SMOTE_RS_AE_Purified_bestmodel.pkl',   'tcga_purified',   True ),
    # ── Non-Purified  Norway ──────────────────────────────────────────────────
    ('NW-NN-RS',             'NN_Norway_RS_bestmodel.pkl',              'norway',          False),
    ('NW-NN-RS-AE',          'NN_Norway_RS_AE_bestmodel.pkl',           'norway',          True ),
    ('NW-NN-IRUS-RS',        'NN_Norway_IRUS_RS_bestmodel.pkl',         'norway',          False),
    ('NW-NN-IRUS-RS-AE',     'NN_Norway_IRUS_RS_AE_bestmodel.pkl',      'norway',          True ),
    ('NW-NN-SMOTE-RS',       'NN_Norway_SMOTE_RS_bestmodel.pkl',        'norway',          False),
    ('NW-SMOTE-RS-AE',       'NN_Norway_SMOTE_RS_AE_bestmodel.pkl',     'norway',          True ),
    # ── Purified  Norway ──────────────────────────────────────────────────────
    ('NW-NN-RS [P]',         'NN_Norway_RS_Purified_bestmodel.pkl',         'norway_purified', False),
    ('NW-NN-RS-AE [P]',      'NN_Norway_RS_AE_Purified_bestmodel.pkl',      'norway_purified', True ),
    ('NW-NN-IRUS-RS [P]',    'NN_Norway_IRUS_RS_Purified_bestmodel.pkl',    'norway_purified', False),
    ('NW-NN-IRUS-RS-AE [P]', 'NN_Norway_IRUS_RS_AE_Purified_bestmodel.pkl', 'norway_purified', True ),
    ('NW-NN-SMOTE-RS [P]',   'NN_Norway_SMOTE_RS_Purified_bestmodel.pkl',   'norway_purified', False),
    ('NW-SMOTE-RS-AE [P]',   'NN_Norway_SMOTE_RS_AE_Purified_bestmodel.pkl','norway_purified', True ),
]

SHAP_BG      = 30   # background samples for SHAP
SHAP_EXPLAIN = 30   # test samples to explain
SHAP_BATCH   = 10   # KernelExplainer batch size
TOP_K        = 10   # top features to report/plot

# Set to a list of labels to restrict SHAP, e.g. ['NN-RS', 'NN-RS-AE']
# Set to None to run SHAP on ALL models
SHAP_RUN = None

for lbl, pkl, ds, ae in MODELS:
    ok = 'ok' if os.path.exists(pkl) else 'MISSING'
    print(f"  {'[AE]' if ae else '    '}  {lbl:<26s}  {ok}")


        NN-RS                       ok
  [AE]  NN-RS-AE                    ok
        NN-IRUS-RS                  ok
  [AE]  NN-IRUS-RS-AE               ok
        NN-SMOTE-RS                 ok
  [AE]  NN-SMOTE-RS-AE              ok
        NN-RS [P]                   ok
  [AE]  NN-RS-AE [P]                ok
        NN-IRUS-RS [P]              ok
  [AE]  NN-IRUS-RS-AE [P]           ok
        NN-SMOTE-RS [P]             ok
  [AE]  NN-SMOTE-RS-AE [P]          ok
        NW-NN-RS                    ok
  [AE]  NW-NN-RS-AE                 ok
        NW-NN-IRUS-RS               ok
  [AE]  NW-NN-IRUS-RS-AE            ok
        NW-NN-SMOTE-RS              ok
  [AE]  NW-SMOTE-RS-AE              ok
        NW-NN-RS [P]                ok
  [AE]  NW-NN-RS-AE [P]             ok
        NW-NN-IRUS-RS [P]           ok
  [AE]  NW-NN-IRUS-RS-AE [P]        ok
        NW-NN-SMOTE-RS [P]          ok
  [AE]  NW-SMOTE-RS-AE [P]          ok


In [3]:
# Cell 3: Load data + autoencoders (decoder from saved model, encoded data from CSVs)
def load_ds(x_tr, x_te, y_tr, y_te):
    le = LabelEncoder()
    Xtr = pd.read_csv(x_tr); Xte = pd.read_csv(x_te)
    ytr = pd.read_csv(y_tr).values.ravel()
    yte = pd.read_csv(y_te).values.ravel()
    return Xtr, Xte, le.fit_transform(ytr), le.transform(yte), le

Xtr_t,Xte_t,ytr_t,yte_t,le_t = load_ds(
    'X_train.csv','X_test.csv','y_train.csv','y_test.csv')
Xtr_n,Xte_n,ytr_n,yte_n,le_n = load_ds(
    'X_train_Norway.csv','X_test_Norway.csv',
    'y_train_Norway.csv','y_test_Norway.csv')
Xtr_tp,Xte_tp,ytr_tp,yte_tp,le_tp = load_ds(
    'X_train_purified.csv','X_test_purified.csv',
    'Y_train_purified.csv','Y_test_purified.csv')
Xtr_np,Xte_np,ytr_np,yte_np,le_np = load_ds(
    'X_train_Norway_purified.csv','X_test_Norway_purified.csv',
    'Y_train_Norway_purified.csv','Y_test_Norway_purified.csv')

DS = {
    'tcga':            dict(Xtr=Xtr_t, Xte=Xte_t, ytr=ytr_t, yte=yte_t, le=le_t,
                            feats=Xtr_t.columns.tolist()),
    'norway':          dict(Xtr=Xtr_n, Xte=Xte_n, ytr=ytr_n, yte=yte_n, le=le_n,
                            feats=Xtr_n.columns.tolist()),
    'tcga_purified':   dict(Xtr=Xtr_tp,Xte=Xte_tp,ytr=ytr_tp,yte=yte_tp,le=le_tp,
                            feats=Xtr_tp.columns.tolist()),
    'norway_purified': dict(Xtr=Xtr_np,Xte=Xte_np,ytr=ytr_np,yte=yte_np,le=le_np,
                            feats=Xtr_np.columns.tolist()),
}

# Load decoder models and pre-encoded CSVs saved by train_autoencoder.ipynb
AE = {}
for dsn, dec_file, enc_tr, enc_te in [
    ('tcga',            'decoder_tcga.keras',            'X_train_AE_tcga.csv',            'X_test_AE_tcga.csv'),
    ('norway',          'decoder_norway.keras',          'X_train_AE_norway.csv',          'X_test_AE_norway.csv'),
    ('tcga_purified',   'decoder_tcga_purified.keras',   'X_train_AE_tcga_purified.csv',   'X_test_AE_tcga_purified.csv'),
    ('norway_purified', 'decoder_Norway_purified.keras', 'X_train_AE_norway_purified.csv', 'X_test_AE_norway_purified.csv'),
]:
    decoder = tf.keras.models.load_model(dec_file)
    Xtr_enc = pd.read_csv(enc_tr).values.astype(np.float32)
    Xte_enc = pd.read_csv(enc_te).values.astype(np.float32)
    AE[dsn] = dict(dec=decoder, Xtr_enc=Xtr_enc, Xte_enc=Xte_enc)
    print(f'[{dsn}] decoder loaded | encoded shapes: train={Xtr_enc.shape}, test={Xte_enc.shape}')

print('Data + autoencoders ready.')

[tcga] decoder loaded | encoded shapes: train=(496, 500), test=(124, 500)
[norway] decoder loaded | encoded shapes: train=(209, 500), test=(52, 500)
[tcga_purified] decoder loaded | encoded shapes: train=(494, 500), test=(123, 500)
[norway_purified] decoder loaded | encoded shapes: train=(218, 500), test=(54, 500)
Data + autoencoders ready.


In [4]:
# Cell 3b: Define model-builder functions required for unpickling
# pickle resolves create_nn / create_classifier by looking them up in __main__,
# so they must exist here with the exact same names used at training time.

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense as _OrigDense

seed = 0  # matches training notebooks

# ── Compatibility patch ────────────────────────────────────────────────────────
# Some purified PKLs were saved with a newer Keras that stores
# `quantization_config` in Dense.get_config().  The current env's Dense
# doesn't accept that kwarg, so strip it on the way in.
_orig_dense_init = _OrigDense.__init__
def _patched_dense_init(self, *args, **kwargs):
    kwargs.pop('quantization_config', None)
    _orig_dense_init(self, *args, **kwargs)
_OrigDense.__init__ = _patched_dense_init
# ──────────────────────────────────────────────────────────────────────────────

def create_nn(meta, neurons_layer1=256, neurons_layer2=128,
              dropout_rate1=0.3, dropout_rate2=0.3,
              learning_rate=0.002, activation='relu'):
    n_features_in = meta["n_features_in_"]
    n_classes_     = meta["n_classes_"]
    model = Sequential([
        Input(shape=(n_features_in,)),
        Dense(neurons_layer1, activation=activation),
        BatchNormalization(),
        Dropout(dropout_rate1),
        Dense(neurons_layer2, activation=activation),
        BatchNormalization(),
        Dropout(dropout_rate2),
        Dense(n_classes_, activation='softmax'),
    ])
    model.compile(optimizer=Adam(learning_rate=learning_rate),
                  loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

# create_classifier references two globals that were in scope during training.
# Set safe defaults here; they will be overridden per-model in the load loop if needed.
X_train_encoded = AE['tcga']['Xtr_enc']          # default: TCGA encoded dim
n_classes        = len(le_t.classes_)             # default: TCGA class count

def create_classifier(hidden_dim=200, dropout_rate1=0.2, dropout_rate2=0.3,
                      learning_rate=2e-5, activation='relu'):
    inp = Input(shape=(X_train_encoded.shape[1],))
    x = BatchNormalization()(inp)
    x = Dropout(dropout_rate1, seed=seed)(x)
    if activation == 'leakyrelu':
        x = Dense(hidden_dim)(x)
        x = LeakyReLU(negative_slope=0.01)(x)
    elif activation == 'relu':
        x = Dense(hidden_dim)(x)
        x = ReLU()(x)
    else:
        x = Dense(hidden_dim, activation='tanh')(x)
    x = BatchNormalization()(x)
    x = Dropout(dropout_rate2, seed=seed)(x)
    out = Dense(n_classes, activation='softmax')(x)
    model = Model(inputs=inp, outputs=out)
    model.compile(optimizer=Adam(learning_rate=learning_rate),
                  loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

print('create_nn and create_classifier defined.')
print('Dense quantization_config patch applied.')


create_nn and create_classifier defined.
Dense quantization_config patch applied.


In [8]:
# Cell 4: Load PKLs + predict + unified PR curves

RESULTS = {}  # label -> {yte, proba, le, is_ae, ds}
PKL     = {}  # label -> model object

for lbl, pkl, dsn, is_ae in MODELS:
    print(f'Loading {pkl} ...', end=' ')
    try:
        # Update globals so create_classifier uses the correct dims if scikeras
        # needs to rebuild the model during deserialization.
        if is_ae:
            globals()['X_train_encoded'] = AE[dsn]['Xtr_enc']
            globals()['n_classes']        = len(DS[dsn]['le'].classes_)

        m = joblib.load(pkl); PKL[lbl] = m
        Xte   = (AE[dsn]['Xte_enc'] if is_ae
                 else DS[dsn]['Xte'].values.astype(np.float32))
        proba = m.predict_proba(Xte)
        le    = DS[dsn]['le']
        yb    = label_binarize(DS[dsn]['yte'], classes=range(len(le.classes_)))
        ap    = average_precision_score(yb, proba, average='micro')
        RESULTS[lbl] = dict(yte=DS[dsn]['yte'], proba=proba,
                             le=le, is_ae=is_ae, ds=dsn)
        print(f'OK  micro-AP={ap:.4f}')
    except Exception as e:
        print(f'FAILED -- {e}')

print(f'\n{len(RESULTS)}/{len(MODELS)} models ready.')

# ── helpers (mirroring evaluation.ipynb exactly) ──────────────────────────────

def pr_curve_micro(r):
    yb = label_binarize(r['yte'], classes=range(len(r['le'].classes_)))
    scores = r['proba']
    precision, recall, _ = precision_recall_curve(yb.ravel(), scores.ravel())
    ap = average_precision_score(yb, scores, average='micro')
    return precision, recall, ap

def plot_pr_curves(ax, entries, title):
    for name, r in entries:
        precision, recall, ap = pr_curve_micro(r)
        ax.plot(recall, precision, label=f"{name} (AP={ap:.3f})")
    ax.set_title(title)
    ax.set_xlabel('Recall')
    ax.set_ylabel('Precision')
    ax.legend(fontsize=8, loc='lower left')

def print_pr_auc(entries, group_name):
    print(f'\n--- {group_name} ---')
    for name, r in entries:
        _, _, ap = pr_curve_micro(r)
        print(f'{name}: PR-AUC (AP) = {ap:.3f}')

def avg_pr_curve(entries, recall_grid=np.linspace(0, 1, 300)):
    prec_list, ap_list = [], []
    for name, r in entries:
        precision, recall, ap = pr_curve_micro(r)
        if recall[0] > recall[-1]:
            recall = recall[::-1]; precision = precision[::-1]
        prec_list.append(np.interp(recall_grid, recall, precision))
        ap_list.append(ap)
    return recall_grid, np.mean(prec_list, axis=0), np.mean(ap_list)

# ── group entries ──────────────────────────────────────────────────────────────
tcga_models            = [(k, v) for k, v in RESULTS.items() if v['ds'] == 'tcga']
tcga_purified_models   = [(k, v) for k, v in RESULTS.items() if v['ds'] == 'tcga_purified']
norway_models          = [(k, v) for k, v in RESULTS.items() if v['ds'] == 'norway']
norway_purified_models = [(k, v) for k, v in RESULTS.items() if v['ds'] == 'norway_purified']

# ── Plot 1: 2×2 grid — TCGA / TCGA Purified / Norway / Norway Purified ───────
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
plot_pr_curves(axes[0, 0], tcga_models,            'TCGA')
plot_pr_curves(axes[0, 1], tcga_purified_models,   'TCGA Purified')
plot_pr_curves(axes[1, 0], norway_models,          'Norway')
plot_pr_curves(axes[1, 1], norway_purified_models, 'Norway Purified')
plt.tight_layout()
plt.savefig(f'{OUT}/pr_tcga_norway.png', dpi=150, bbox_inches='tight')
plt.show(); plt.close()
print('Saved pr_tcga_norway.png')

# ── Print PR-AUC ───────────────────────────────────────────────────────────────
print_pr_auc(tcga_models,            'TCGA')
print_pr_auc(tcga_purified_models,   'TCGA Purified')
print_pr_auc(norway_models,          'Norway')
print_pr_auc(norway_purified_models, 'Norway Purified')

# ── Per-class PR-AUC ──────────────────────────────────────────────────────────
def per_class_pr_auc(r):
    yb = label_binarize(r['yte'], classes=range(len(r['le'].classes_)))
    ap_dict = {}
    for i, cls in enumerate(r['le'].classes_):
        y_true = yb[:, i]
        if y_true.sum() == 0 or y_true.sum() == len(y_true):
            ap_dict[cls] = np.nan
        else:
            ap_dict[cls] = average_precision_score(y_true, r['proba'][:, i])
    return ap_dict

def print_per_class_pr_auc(entries, group_name):
    print(f'\n--- {group_name} ---')
    for name, r in entries:
        ap_dict = per_class_pr_auc(r)
        print(name)
        for cls, ap in ap_dict.items():
            print(f'  {cls}: {ap:.3f}' if not np.isnan(ap) else f'  {cls}: nan')

print_per_class_pr_auc(tcga_models,            'TCGA')
print_per_class_pr_auc(tcga_purified_models,   'TCGA Purified')
print_per_class_pr_auc(norway_models,          'Norway')
print_per_class_pr_auc(norway_purified_models, 'Norway Purified')

# ── Plot 2: combined single-axes (group-colored, varying linestyles) ───────────
import itertools

group_colors = {
    'TCGA':            'tab:blue',
    'TCGA Purified':   'tab:cyan',
    'Norway':          'tab:green',
    'Norway Purified': 'tab:olive',
}
groups_all = [
    ('TCGA',            tcga_models),
    ('TCGA Purified',   tcga_purified_models),
    ('Norway',          norway_models),
    ('Norway Purified', norway_purified_models),
]

fig, ax = plt.subplots(figsize=(13, 9))
for gname, entries in groups_all:
    ls_cycle = itertools.cycle(['-', '--', '-.', ':', (0, (3,1,1,1)), (0, (5,1))])
    for name, r in entries:
        precision, recall, ap = pr_curve_micro(r)
        ax.plot(recall, precision,
                color=group_colors[gname], linestyle=next(ls_cycle),
                linewidth=1.6, alpha=0.9,
                label=f"{gname}: {name} (AP={ap:.3f})")
ax.set_title('PR Curves (Micro-Average) — All Models')
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=7)
plt.tight_layout()
plt.savefig(f'{OUT}/pr_all_combined.png', dpi=150, bbox_inches='tight')
plt.show(); plt.close()
print('Saved pr_all_combined.png')

# ── Plot 3: averaged PR curve per group ───────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 7))
for gname, entries in groups_all:
    if not entries:
        continue
    rgrid, mean_prec, mean_ap = avg_pr_curve(entries)
    ax.plot(rgrid, mean_prec, color=group_colors[gname],
            linewidth=2.5, label=f"{gname} (mean AP={mean_ap:.3f})")
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Average PR Curves (Micro-Average) per Dataset Group')
ax.legend(loc='lower left')
plt.tight_layout()
plt.savefig(f'{OUT}/pr_avg_per_group.png', dpi=150, bbox_inches='tight')
plt.show(); plt.close()
print('Saved pr_avg_per_group.png')

# ── Plot 4: unified — TCGA vs TCGA Purified and Norway vs Norway Purified ─────
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
for ax, (g1, e1), (g2, e2), title in [
    (axes[0],
     ('TCGA',          tcga_models),
     ('TCGA Purified', tcga_purified_models),
     'TCGA vs TCGA Purified'),
    (axes[1],
     ('Norway',          norway_models),
     ('Norway Purified', norway_purified_models),
     'Norway vs Norway Purified'),
]:
    for gname, entries in [(g1, e1), (g2, e2)]:
        if not entries:
            continue
        ls_cycle = itertools.cycle(['-', '--', '-.', ':', (0, (3,1,1,1)), (0, (5,1))])
        for name, r in entries:
            precision, recall, ap = pr_curve_micro(r)
            ax.plot(recall, precision,
                    color=group_colors[gname], linestyle=next(ls_cycle),
                    linewidth=1.4, alpha=0.85,
                    label=f"{gname}: {name} (AP={ap:.3f})")
        if entries:
            rgrid, mean_prec, mean_ap = avg_pr_curve(entries)
            ax.plot(rgrid, mean_prec,
                    color=group_colors[gname], linestyle='-',
                    linewidth=3.0, alpha=1.0,
                    label=f"{gname} MEAN (AP={mean_ap:.3f})")
    ax.set_title(title)
    ax.set_xlabel('Recall')
    ax.set_ylabel('Precision')
    ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=7)
plt.tight_layout()
plt.savefig(f'{OUT}/pr_unified_comparison.png', dpi=150, bbox_inches='tight')
plt.show(); plt.close()
print('Saved pr_unified_comparison.png')


Loading NN_RS_bestmodel.pkl ... OK  micro-AP=0.8307
Loading NN_RS_AE_bestmodel.pkl ... OK  micro-AP=0.8643
Loading NN_IRUS_RS_bestmodel.pkl ... OK  micro-AP=0.7362
Loading NN_IRUS_RS_AE_bestmodel.pkl ... OK  micro-AP=0.7267
Loading NN_SMOTE_RS_bestmodel.pkl ... OK  micro-AP=0.8108
Loading NN_SMOTE_RS_AE_bestmodel.pkl ... OK  micro-AP=0.8246
Loading NN_RS_Purified_bestmodel.pkl ... OK  micro-AP=0.8135
Loading NN_RS_AE_Purified_bestmodel.pkl ... OK  micro-AP=0.8655
Loading NN_IRUS_RS_Purified_bestmodel.pkl ... OK  micro-AP=0.7957
Loading NN_IRUS_RS_AE_Purified_bestmodel.pkl ... OK  micro-AP=0.7748
Loading NN_SMOTE_RS_Purified_bestmodel.pkl ... OK  micro-AP=0.8366
Loading NN_SMOTE_RS_AE_Purified_bestmodel.pkl ... OK  micro-AP=0.2638
Loading NN_Norway_RS_bestmodel.pkl ... OK  micro-AP=0.8681
Loading NN_Norway_RS_AE_bestmodel.pkl ... OK  micro-AP=0.7588
Loading NN_Norway_IRUS_RS_bestmodel.pkl ... OK  micro-AP=0.5396
Loading NN_Norway_IRUS_RS_AE_bestmodel.pkl ... OK  micro-AP=0.5989
Loading 

In [ ]:
# Cell 5: SHAP (per-model) + accumulate for overall plots

targets = SHAP_RUN if SHAP_RUN else [lbl for lbl, *_ in MODELS]

# Accumulators for overall SHAP (gene / original feature space, 3-D: [n, feat, classes])
SHAP_ACCUM = {
    'tcga':            {'sv_list': [], 'X_list': [], 'feats': None},
    'norway':          {'sv_list': [], 'X_list': [], 'feats': None},
    'tcga_purified':   {'sv_list': [], 'X_list': [], 'feats': None},
    'norway_purified': {'sv_list': [], 'X_list': [], 'feats': None},
}

for lbl, pkl, dsn, is_ae in MODELS:
    if lbl not in targets or lbl not in RESULTS or lbl not in PKL:
        continue

    sep = '-' * 60
    print(f'\n{sep}')
    print(f"SHAP: {lbl}  ({'AE' if is_ae else 'regular'}, {dsn.upper()})")
    print(sep)

    mdir = os.path.join(OUT, 'shap', lbl.replace(' ', '_'))
    os.makedirs(mdir, exist_ok=True)

    ds = DS[dsn]; le = ds['le']; classes = le.classes_.tolist()
    model = PKL[lbl]; km = get_keras_model(model)
    if km is None:
        print('  Cannot extract Keras model -- skip'); continue

    if is_ae:
        Xtr_s  = AE[dsn]['Xtr_enc'].astype(np.float32)
        Xte_s  = AE[dsn]['Xte_enc'].astype(np.float32)
        fnames = [f'Enc_{i}' for i in range(Xtr_s.shape[1])]
        dec    = AE[dsn]['dec']
    else:
        Xtr_s  = ds['Xtr'].values.astype(np.float32)
        Xte_s  = ds['Xte'].values.astype(np.float32)
        fnames = ds['feats']
        dec    = None

    rng = np.random.RandomState(42)
    bg  = Xtr_s[rng.choice(len(Xtr_s), min(SHAP_BG,     len(Xtr_s)), replace=False)]
    te  = Xte_s[rng.choice(len(Xte_s), min(SHAP_EXPLAIN, len(Xte_s)), replace=False)]

    # compute SHAP values
    sv = None
    try:
        expl = shap.DeepExplainer(km, bg)
        sv   = expl.shap_values(te, check_additivity=False)
        del expl; gc.collect()
        print('  DeepExplainer OK')
    except Exception as e:
        print(f'  DeepExplainer failed ({e}) -> KernelExplainer')
        try:
            expl = shap.KernelExplainer(
                lambda X: km.predict(X, batch_size=32, verbose=0), bg)
            batches = [expl.shap_values(te[i:i+SHAP_BATCH])
                       for i in range(0, len(te), SHAP_BATCH)]
            gc.collect()
            if isinstance(batches[0], list):
                sv = [np.vstack([b[c] for b in batches])
                      for c in range(len(batches[0]))]
            else:
                sv = np.vstack(batches)
            del expl; gc.collect()
            print('  KernelExplainer OK')
        except Exception as e2:
            print(f'  KernelExplainer also failed ({e2}) -- skip')
            continue

    if isinstance(sv, list):
        sv = np.stack([np.array(s) for s in sv], axis=2)
    print(f'  SHAP shape: {sv.shape}')
    sv_ov = np.abs(sv).max(axis=2) if sv.ndim == 3 else np.abs(sv)

    # per-class plots in encoded (or original) space
    if sv.ndim == 3:
        for ci, cls in enumerate(classes[:sv.shape[2]]):
            sv_c = sv[:, :, ci]
            imp  = (pd.DataFrame({'Feature': fnames,
                                  'MeanAbsSHAP': np.abs(sv_c).mean(0)})
                    .sort_values('MeanAbsSHAP', ascending=False))
            imp.head(TOP_K).to_csv(f'{mdir}/top_{cls.lower()}.csv', index=False)

            plt.figure(figsize=(11, 7))
            shap.summary_plot(sv_c, te, feature_names=fnames,
                              max_display=TOP_K, show=False)
            plt.title(f'SHAP -- {cls} | {lbl}', fontsize=13, pad=18)
            plt.tight_layout()
            plt.savefig(f'{mdir}/summary_{cls.lower()}.png',
                        dpi=150, bbox_inches='tight')
            plt.close()

            plt.figure(figsize=(10, 7))
            shap.summary_plot(sv_c, te, feature_names=fnames,
                              max_display=TOP_K, plot_type='bar', show=False)
            plt.title(f'SHAP Bar -- {cls} | {lbl}', fontsize=13, pad=18)
            plt.tight_layout()
            plt.savefig(f'{mdir}/bar_{cls.lower()}.png',
                        dpi=150, bbox_inches='tight')
            plt.close()

    # decode to original gene space (AE models only)
    sv_gene_3d = None   # [n_te, n_gene, n_class] in original feature space
    X_gene     = None

    if is_ae and dec is not None:
        print('  Decoding SHAP to original gene space via trained decoder...')
        Xte_orig = ds['Xte'].values[:len(te)]

        # overall decoded bar
        dec_ov = np.abs(dec.predict(sv_ov.astype(np.float32),
                                    batch_size=32, verbose=0))
        dec_df = (pd.DataFrame({'Feature': ds['feats'],
                                'DecodedSHAP': dec_ov.mean(0)})
                  .sort_values('DecodedSHAP', ascending=False))
        dec_df.to_csv(f'{mdir}/decoded_overall.csv', index=False)

        plt.figure(figsize=(10, 7))
        sns.barplot(data=dec_df.head(TOP_K), x='DecodedSHAP', y='Feature',
                    hue='Feature', palette='Blues_d', legend=False)
        plt.title(f'Decoded SHAP -- Original Genes | {lbl}', fontsize=13)
        plt.tight_layout()
        plt.savefig(f'{mdir}/decoded_bar_overall.png',
                    dpi=150, bbox_inches='tight')
        plt.close()

        # per-class decoded
        if sv.ndim == 3:
            sv_dec_classes = []
            for ci, cls in enumerate(classes[:sv.shape[2]]):
                sv_dec = np.abs(dec.predict(
                    np.abs(sv[:, :, ci]).astype(np.float32),
                    batch_size=32, verbose=0))          # [n_te, n_gene]
                sv_dec_classes.append(sv_dec)

                cls_df = (pd.DataFrame({'Feature': ds['feats'],
                                        'DecodedSHAP': sv_dec.mean(0)})
                          .sort_values('DecodedSHAP', ascending=False))
                cls_df.head(TOP_K).to_csv(
                    f'{mdir}/decoded_top_{cls.lower()}.csv', index=False)

                plt.figure(figsize=(12, 8))
                shap.summary_plot(sv_dec, Xte_orig, feature_names=ds['feats'],
                                  max_display=TOP_K, show=False)
                plt.title(f'Decoded SHAP -- {cls} | {lbl}', fontsize=13, pad=18)
                plt.tight_layout()
                plt.savefig(f'{mdir}/decoded_summary_{cls.lower()}.png',
                            dpi=150, bbox_inches='tight')
                plt.close()

                plt.figure(figsize=(10, 7))
                shap.summary_plot(sv_dec, Xte_orig, feature_names=ds['feats'],
                                  max_display=TOP_K, plot_type='bar', show=False)
                plt.title(f'Decoded SHAP Bar -- {cls} | {lbl}',
                          fontsize=13, pad=18)
                plt.tight_layout()
                plt.savefig(f'{mdir}/decoded_bar_{cls.lower()}.png',
                            dpi=150, bbox_inches='tight')
                plt.close()

            # stack decoded classes → [n_te, n_gene, n_class]
            sv_gene_3d = np.stack(sv_dec_classes, axis=2)
            X_gene     = Xte_orig.astype(np.float32)

        print(f'  Decoded results saved -> {mdir}')

    elif not is_ae and sv.ndim == 3:
        # Already in gene space
        sv_gene_3d = sv                          # [n_te, n_gene, n_class]
        X_gene     = te.astype(np.float32)

    # ── accumulate for overall SHAP ───────────────────────────────────────────
    if sv_gene_3d is not None:
        SHAP_ACCUM[dsn]['sv_list'].append(sv_gene_3d)
        SHAP_ACCUM[dsn]['X_list'].append(X_gene)
        if SHAP_ACCUM[dsn]['feats'] is None:
            SHAP_ACCUM[dsn]['feats'] = ds['feats']

    gc.collect()

print(f'\nDone.  All outputs -> {OUT}/')



------------------------------------------------------------
SHAP: NN-RS  (regular, TCGA)
------------------------------------------------------------
  DeepExplainer OK
  SHAP shape: (30, 19277, 5)

------------------------------------------------------------
SHAP: NN-RS-AE  (AE, TCGA)
------------------------------------------------------------
  DeepExplainer OK
  SHAP shape: (30, 500, 5)
  Decoding SHAP to original gene space via trained decoder...
  Decoded results saved -> pr_shap_out_Purified\shap\NN-RS-AE

------------------------------------------------------------
SHAP: NN-IRUS-RS  (regular, TCGA)
------------------------------------------------------------
  DeepExplainer OK
  SHAP shape: (30, 19277, 5)

------------------------------------------------------------
SHAP: NN-IRUS-RS-AE  (AE, TCGA)
------------------------------------------------------------
  DeepExplainer failed (in user code:

    File "C:\Users\mike\AppData\Roaming\Python\Python311\site-packages\shap\expla

100%|██████████| 10/10 [00:29<00:00,  2.99s/it]


  KernelExplainer OK
  SHAP shape: (30, 500, 5)
  Decoding SHAP to original gene space via trained decoder...
  Decoded results saved -> pr_shap_out_Purified\shap\NN-IRUS-RS-AE

------------------------------------------------------------
SHAP: NN-SMOTE-RS  (regular, TCGA)
------------------------------------------------------------
  DeepExplainer OK
  SHAP shape: (30, 19277, 5)

------------------------------------------------------------
SHAP: NN-SMOTE-RS-AE  (AE, TCGA)
------------------------------------------------------------
  DeepExplainer OK
  SHAP shape: (30, 500, 5)
  Decoding SHAP to original gene space via trained decoder...
  Decoded results saved -> pr_shap_out_Purified\shap\NN-SMOTE-RS-AE

------------------------------------------------------------
SHAP: NN-RS [P]  (regular, TCGA_PURIFIED)
------------------------------------------------------------
  DeepExplainer OK
  SHAP shape: (30, 19277, 5)

------------------------------------------------------------
SHAP: NN-R

KeyboardInterrupt: 

: 

In [7]:
# Cell 6: Overall SHAP — 1 bar chart + 1 summary plot per dataset
# sv_gene_3d shape: [n_samples, n_genes, n_classes]
# Follows the same pattern as evaluation.ipynb:
#   sv_overall = np.abs(shap_values).max(axis=2)  →  [n, n_gene]

for dsn in ['tcga', 'norway', 'tcga_purified', 'norway_purified']:
    if not SHAP_ACCUM[dsn]['sv_list']:
        print(f'No SHAP data accumulated for {dsn.upper()} — skipping overall plot.')
        continue

    sv_agg = np.vstack(SHAP_ACCUM[dsn]['sv_list'])   # [N, n_gene, n_class]
    X_agg  = np.vstack(SHAP_ACCUM[dsn]['X_list'])    # [N, n_gene]
    feats  = SHAP_ACCUM[dsn]['feats']

    # overall: max |SHAP| across classes per sample-feature
    sv_overall = np.abs(sv_agg).max(axis=2)           # [N, n_gene]

    # mean importance per feature for bar chart
    mean_abs_shap = sv_overall.mean(axis=0)
    imp_df = (pd.DataFrame({'Feature': feats,
                            'MeanAbsSHAP_MaxAcrossSubtypes': mean_abs_shap})
              .sort_values('MeanAbsSHAP_MaxAcrossSubtypes', ascending=False))

    print(f'\n=== Overall Top {TOP_K} Features — {dsn.upper()} ===')
    print(imp_df.head(TOP_K).to_string(index=False))

    # ── Bar chart ─────────────────────────────────────────────────────────────
    plt.figure(figsize=(10, 7))
    sns.barplot(data=imp_df.head(TOP_K),
                x='MeanAbsSHAP_MaxAcrossSubtypes', y='Feature',
                hue='Feature', palette='Blues_d', legend=False)
    plt.title(f'Overall SHAP Feature Importance — {dsn.upper()} (All Models)',
              fontsize=13)
    plt.xlabel('Mean |SHAP| Value (Max Across Subtypes)')
    plt.ylabel('Gene')
    plt.tight_layout()
    plt.savefig(f'{OUT}/overall_shap_bar_{dsn}.png', dpi=150, bbox_inches='tight')
    plt.show(); plt.close()
    print(f'Saved overall_shap_bar_{dsn}.png')

    # ── Summary (beeswarm) ────────────────────────────────────────────────────
    plt.figure(figsize=(11, 7))
    shap.summary_plot(sv_overall, X_agg, feature_names=feats,
                      max_display=20, show=False)
    plt.title(f'Overall SHAP Summary — {dsn.upper()} (All Models)',
              fontsize=13, pad=18)
    plt.tight_layout()
    plt.savefig(f'{OUT}/overall_shap_summary_{dsn}.png', dpi=150, bbox_inches='tight')
    plt.show(); plt.close()
    print(f'Saved overall_shap_summary_{dsn}.png')

    imp_df.to_csv(f'{OUT}/overall_shap_importance_{dsn}.csv', index=False)

gc.collect()
print('\nOverall SHAP done.')



=== Overall Top 10 Features — TCGA ===
 Feature  MeanAbsSHAP_MaxAcrossSubtypes
  DLEU2L                       0.009731
   PRINS                       0.008677
  ZNF732                       0.008563
  AKR1D1                       0.008515
C19orf18                       0.008328
SERPINB9                       0.008019
VTRNA2-1                       0.007829
 SNORD17                       0.007806
 SNORA2B                       0.007764
 ARHGDIB                       0.007495
Saved overall_shap_bar_tcga.png
Saved overall_shap_summary_tcga.png

=== Overall Top 10 Features — NORWAY ===
  Feature  MeanAbsSHAP_MaxAcrossSubtypes
      PRL                       0.017449
  FILIP1L                       0.016894
  SNORD57                       0.016216
    AMY2B                       0.016055
    CPT1B                       0.016044
    TRIB3                       0.015933
  B3GALT5                       0.015910
  SNORD98                       0.015886
  PPP2R2D                       0.015725


In [ ]:

# Cell 7: Friedman + Nemenyi — compare 6 configurations simultaneously
#
# Setup: 6 "treatment" configurations × n "block" observations
#   Per-dataset analysis : 5 blocks  (one per subtype class)
#   Pooled analysis      : 20 blocks (4 dataset-variants × 5 classes)
#
# Nemenyi post-hoc heatmap saved when Friedman is significant.

import scipy.stats as ss

try:
    import scikit_posthocs as sp
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'scikit-posthocs', '-q'])
    import scikit_posthocs as sp

# ── 1. Map every model label → base configuration ────────────────────────────
config_map = {
    'NN-RS':               'RS',          'NW-NN-RS':              'RS',
    'NN-RS [P]':           'RS',          'NW-NN-RS [P]':          'RS',
    'NN-RS-AE':            'RS-AE',       'NW-NN-RS-AE':           'RS-AE',
    'NN-RS-AE [P]':        'RS-AE',       'NW-NN-RS-AE [P]':       'RS-AE',
    'NN-IRUS-RS':          'IRUS-RS',     'NW-NN-IRUS-RS':         'IRUS-RS',
    'NN-IRUS-RS [P]':      'IRUS-RS',     'NW-NN-IRUS-RS [P]':     'IRUS-RS',
    'NN-IRUS-RS-AE':       'IRUS-RS-AE',  'NW-NN-IRUS-RS-AE':      'IRUS-RS-AE',
    'NN-IRUS-RS-AE [P]':   'IRUS-RS-AE',  'NW-NN-IRUS-RS-AE [P]':  'IRUS-RS-AE',
    'NN-SMOTE-RS':         'SMOTE-RS',    'NW-NN-SMOTE-RS':        'SMOTE-RS',
    'NN-SMOTE-RS [P]':     'SMOTE-RS',    'NW-NN-SMOTE-RS [P]':    'SMOTE-RS',
    'NN-SMOTE-RS-AE':      'SMOTE-RS-AE', 'NW-SMOTE-RS-AE':        'SMOTE-RS-AE',
    'NN-SMOTE-RS-AE [P]':  'SMOTE-RS-AE', 'NW-SMOTE-RS-AE [P]':    'SMOTE-RS-AE',
}
CONFIGS = ['RS', 'RS-AE', 'IRUS-RS', 'IRUS-RS-AE', 'SMOTE-RS', 'SMOTE-RS-AE']

# ── 2. Build per-class AP DataFrame ─────────────────────────────────────────
records = []
for lbl, r in RESULTS.items():
    ap_dict = per_class_pr_auc(r)
    cfg  = config_map.get(lbl, lbl)
    dsn  = r['ds']
    purified = '_purified' in dsn
    ds_base  = 'norway' if 'norway' in dsn else 'tcga'
    for cls, ap in ap_dict.items():
        if not np.isnan(ap):
            records.append(dict(
                label=lbl, config=cfg, ds=dsn, ds_base=ds_base,
                purified=purified, cls=str(cls), ap=ap
            ))

ap_df = pd.DataFrame(records)
ap_df['block'] = ap_df['ds'] + '_' + ap_df['cls']   # unique (dataset, class) block id
print(f'AP table: {len(ap_df)} rows')
print(f'Configs : {CONFIGS}')
print(f'Datasets: {sorted(ap_df["ds"].unique())}')
print()

# ── 3. Friedman + Nemenyi helper ─────────────────────────────────────────────
def run_friedman_nemenyi(pivot_df, title, save_prefix=None):
    """
    pivot_df : DataFrame with blocks as index, configs as columns.
    Returns (nemenyi_pval_df | None, mean_ranks_series).
    """
    pivot_clean = pivot_df.reindex(columns=CONFIGS).dropna()
    n_blk, n_cfg = pivot_clean.shape
    print(f'\n{"="*62}')
    print(f'  {title}')
    print(f'  blocks={n_blk}, treatments={n_cfg}')

    if n_blk < 3:
        print('  *** Too few blocks — skip ***')
        return None, None

    stat, p = ss.friedmanchisquare(*[pivot_clean[c].values for c in CONFIGS])
    sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
    print(f'  Friedman χ²={stat:.4f},  p={p:.4f}  {sig}')

    # Mean ranks (lower rank = better)
    ranks      = pivot_clean.rank(axis=1, ascending=False)
    mean_ranks = ranks.mean().sort_values()
    print('  Mean ranks (1=best):')
    for cfg, rank in mean_ranks.items():
        print(f'    {cfg:<16s}: {rank:.3f}  (mean AP={pivot_clean[cfg].mean():.3f})')

    nemenyi_p = None
    if p < 0.05:
        nemenyi_p = sp.posthoc_nemenyi_friedman(pivot_clean.values)
        nemenyi_p.index   = CONFIGS
        nemenyi_p.columns = CONFIGS

        sig_pairs = [(ci, cj, nemenyi_p.loc[ci, cj])
                     for i, ci in enumerate(CONFIGS)
                     for j, cj in enumerate(CONFIGS)
                     if i < j and nemenyi_p.loc[ci, cj] < 0.05]
        if sig_pairs:
            print('  Nemenyi significant pairs (p < 0.05):')
            for ci, cj, pv in sorted(sig_pairs, key=lambda x: x[2]):
                print(f'    {ci:<16s} vs {cj:<16s}: p={pv:.4f}')
        else:
            print('  Nemenyi: no pairs below p=0.05 after post-hoc')

        # ── Heatmap ──────────────────────────────────────────────────────────
        if save_prefix:
            mask_upper = np.triu(np.ones(nemenyi_p.shape, dtype=bool), k=0)
            fig, axes = plt.subplots(1, 2, figsize=(14, 5),
                                     gridspec_kw={'width_ratios': [2, 1]})

            # Left: p-value heatmap (lower triangle only)
            ax = axes[0]
            annot_mat = nemenyi_p.copy().astype(str)
            for ci in CONFIGS:
                for cj in CONFIGS:
                    v = nemenyi_p.loc[ci, cj]
                    star = '***' if v < 0.001 else '**' if v < 0.01 else '*' if v < 0.05 else ''
                    annot_mat.loc[ci, cj] = f'{v:.3f}{star}'

            sns.heatmap(nemenyi_p, mask=mask_upper, annot=annot_mat, fmt='',
                        cmap='RdYlGn_r', vmin=0, vmax=1,
                        linewidths=0.5, ax=ax,
                        cbar_kws={'label': 'p-value'})
            ax.set_title(f'Nemenyi Post-Hoc p-values\n{title}', fontsize=10)
            ax.set_xticklabels(ax.get_xticklabels(), rotation=35, ha='right', fontsize=9)
            ax.set_yticklabels(ax.get_yticklabels(), fontsize=9)

            # Right: mean rank bar chart
            ax2 = axes[1]
            mean_ranks_sorted = mean_ranks.sort_values()
            colors_bar = ['#2ecc71' if r == mean_ranks_sorted.min() else '#3498db'
                          for r in mean_ranks_sorted]
            ax2.barh(mean_ranks_sorted.index, mean_ranks_sorted.values,
                     color=colors_bar, edgecolor='gray', linewidth=0.6)
            ax2.set_xlabel('Mean Rank (lower = better)')
            ax2.set_title('Mean Friedman Rank', fontsize=10)
            ax2.invert_yaxis()
            ax2.set_xlim(0, len(CONFIGS) + 0.5)
            ax2.axvline(x=mean_ranks_sorted.min(), color='green', ls='--', lw=1, alpha=0.7)
            for bar_y, val in zip(range(len(mean_ranks_sorted)), mean_ranks_sorted.values):
                ax2.text(val + 0.05, bar_y, f'{val:.2f}', va='center', fontsize=9)

            plt.suptitle(title, fontsize=11, y=1.02)
            plt.tight_layout()
            fname = f'{OUT}/friedman_nemenyi_{save_prefix}.png'
            plt.savefig(fname, dpi=150, bbox_inches='tight')
            plt.show(); plt.close()
            print(f'  Saved {fname}')
    else:
        print('  Friedman not significant — skipping Nemenyi post-hoc.')

    return nemenyi_p, mean_ranks

# ── 4. Run Friedman per dataset group ────────────────────────────────────────
friedman_results = {}
for dsn in ['tcga', 'tcga_purified', 'norway', 'norway_purified']:
    sub   = ap_df[ap_df['ds'] == dsn]
    pivot = sub.pivot_table(index='cls', columns='config', values='ap', aggfunc='mean')
    nem, ranks = run_friedman_nemenyi(pivot, f'Friedman — {dsn.upper()}', save_prefix=dsn)
    friedman_results[dsn] = {'nemenyi': nem, 'mean_ranks': ranks}

# ── 5. Pooled Friedman (4 datasets × 5 classes = 20 blocks) ─────────────────
pivot_pooled = ap_df.pivot_table(index='block', columns='config', values='ap', aggfunc='mean')
nem_pooled, ranks_pooled = run_friedman_nemenyi(
    pivot_pooled,
    'Friedman — POOLED (4 datasets × 5 classes = 20 blocks)',
    save_prefix='pooled'
)
friedman_results['pooled'] = {'nemenyi': nem_pooled, 'mean_ranks': ranks_pooled}

print('\nFriedman + Nemenyi done.')


In [ ]:

# Cell 8: Wilcoxon signed-rank tests
#
# Two comparisons of interest (using per-class AP as the repeated measure):
#   A) AE vs No-AE  — does the autoencoder pre-training help?
#      Pairs: (RS, RS-AE), (IRUS-RS, IRUS-RS-AE), (SMOTE-RS, SMOTE-RS-AE)
#      Matched on: (dataset × class) block (up to 20 pairs each)
#
#   B) Raw vs Purified  — does tumour purity filtering change performance?
#      Matched on: (config × class) block (up to 30 pairs per cohort)

from scipy.stats import wilcoxon

def _sig_star(p):
    return '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'

def wilcoxon_paired(vec_a, vec_b, label_a, label_b, context=''):
    """Run Wilcoxon on two aligned AP vectors. Returns a result dict."""
    assert len(vec_a) == len(vec_b)
    diff = np.array(vec_a) - np.array(vec_b)
    if np.all(diff == 0):
        print(f'  {context}: all differences are zero — skip')
        return None
    try:
        stat, p = wilcoxon(vec_a, vec_b, alternative='two-sided')
    except Exception as e:
        print(f'  {context}: Wilcoxon failed ({e})')
        return None
    direction = f'{label_a} > {label_b}' if np.mean(vec_a) > np.mean(vec_b) else f'{label_b} > {label_a}'
    print(f'  {context:<42s}  W={stat:6.1f}  p={p:.4f} {_sig_star(p)}'
          f'  [{label_a}={np.mean(vec_a):.3f}  {label_b}={np.mean(vec_b):.3f}  → {direction}]'
          f'  n={len(vec_a)}')
    return dict(context=context, W=stat, p=p, sig=_sig_star(p),
                mean_a=np.mean(vec_a), mean_b=np.mean(vec_b), n=len(vec_a))

wilcoxon_rows = []   # collect all results for a summary table

# ── A) AE vs No-AE ────────────────────────────────────────────────────────────
print('='*72)
print('Wilcoxon: AE vs No-AE')
print('  (matched on dataset × class block; 20 pairs per base config)')
print('='*72)

ae_pairs = [('RS', 'RS-AE'), ('IRUS-RS', 'IRUS-RS-AE'), ('SMOTE-RS', 'SMOTE-RS-AE')]

all_no_ae, all_ae = [], []
for cfg_base, cfg_ae in ae_pairs:
    sub_a = ap_df[ap_df['config'] == cfg_base ].set_index('block')['ap']
    sub_b = ap_df[ap_df['config'] == cfg_ae   ].set_index('block')['ap']
    common = sub_a.index.intersection(sub_b.index)
    if len(common) < 5:
        print(f'  {cfg_base} vs {cfg_ae}: too few matched blocks ({len(common)}) — skip')
        continue
    va, vb = sub_a[common].values, sub_b[common].values
    all_no_ae.extend(va); all_ae.extend(vb)
    res = wilcoxon_paired(va, vb, cfg_base, cfg_ae,
                          context=f'{cfg_base} vs {cfg_ae}')
    if res: wilcoxon_rows.append(res)

# Pooled AE test (all No-AE vs all AE, 60 pairs)
if len(all_no_ae) >= 5:
    print()
    res = wilcoxon_paired(all_no_ae, all_ae, 'No-AE (all)', 'AE (all)',
                          context='ALL No-AE vs ALL AE (pooled)')
    if res: wilcoxon_rows.append(res)

# Per-cohort (TCGA / Norway only)
print()
for cohort in ['tcga', 'norway']:
    sub = ap_df[ap_df['ds_base'] == cohort]
    for cfg_base, cfg_ae in ae_pairs:
        sa = sub[sub['config'] == cfg_base].set_index('block')['ap']
        sb = sub[sub['config'] == cfg_ae  ].set_index('block')['ap']
        common = sa.index.intersection(sb.index)
        if len(common) < 3:
            continue
        res = wilcoxon_paired(sa[common].values, sb[common].values,
                              cfg_base, cfg_ae,
                              context=f'[{cohort.upper()}] {cfg_base} vs {cfg_ae}')
        if res: wilcoxon_rows.append(res)

# ── B) Raw vs Purified ────────────────────────────────────────────────────────
print()
print('='*72)
print('Wilcoxon: Raw vs Purified  (matched on config × class; 30 pairs per cohort)')
print('='*72)

for cohort in ['tcga', 'norway']:
    sub_raw = ap_df[(ap_df['ds_base'] == cohort) & (~ap_df['purified'])]
    sub_pur = ap_df[(ap_df['ds_base'] == cohort) & ( ap_df['purified'])]
    # Match by config × class
    sub_raw = sub_raw.set_index(['config', 'cls'])['ap']
    sub_pur = sub_pur.set_index(['config', 'cls'])['ap']
    common  = sub_raw.index.intersection(sub_pur.index)
    if len(common) < 5:
        print(f'  [{cohort.upper()}] Raw vs Purified: too few matched pairs ({len(common)}) — skip')
        continue
    va, vb = sub_raw[common].values, sub_pur[common].values
    res = wilcoxon_paired(va, vb, f'{cohort.upper()}-Raw', f'{cohort.upper()}-Purified',
                          context=f'[{cohort.upper()}] Raw vs Purified')
    if res: wilcoxon_rows.append(res)

    # Also per-config breakdown
    for cfg in CONFIGS:
        sa = ap_df[(ap_df['ds_base'] == cohort) & (~ap_df['purified']) & (ap_df['config'] == cfg)]
        sb = ap_df[(ap_df['ds_base'] == cohort) & ( ap_df['purified']) & (ap_df['config'] == cfg)]
        sa = sa.set_index('cls')['ap']; sb = sb.set_index('cls')['ap']
        common_c = sa.index.intersection(sb.index)
        if len(common_c) < 3:
            continue
        res = wilcoxon_paired(sa[common_c].values, sb[common_c].values,
                              'Raw', 'Purified',
                              context=f'[{cohort.upper()}] {cfg} Raw vs Pur')
        if res: wilcoxon_rows.append(res)

# ── Summary table ─────────────────────────────────────────────────────────────
if wilcoxon_rows:
    wdf = pd.DataFrame(wilcoxon_rows)
    wdf = wdf[['context', 'n', 'W', 'p', 'sig', 'mean_a', 'mean_b']]
    wdf.columns = ['Comparison', 'n_pairs', 'W', 'p-value', 'sig', 'mean_A', 'mean_B']
    wdf = wdf.round({'W': 1, 'p-value': 4, 'mean_A': 3, 'mean_B': 3})
    csv_path = f'{OUT}/wilcoxon_summary.csv'
    wdf.to_csv(csv_path, index=False)
    print(f'\nWilcoxon summary saved → {csv_path}')
    print(wdf.to_string(index=False))

print('\nWilcoxon tests done.')


In [ ]:

# Cell 9: Bootstrap 95% CI on Micro-Average PR-AUC
#
# Motivation: Norway test set is small (n=52), so point-estimate AP values
# carry substantial sampling uncertainty. Bootstrap CI quantifies this.
#
# Method: resample test indices with replacement 2000 times, recompute
# micro-average AP each time, report 2.5th–97.5th percentile as 95% CI.

N_BOOT = 2000

def bootstrap_micro_ap(r, n_boot=N_BOOT, seed=42):
    """Bootstrap CI for micro-average AP. Returns (point, lo, hi, boot_samples)."""
    rng  = np.random.RandomState(seed)
    yte  = r['yte']
    prob = r['proba']
    n_cls = prob.shape[1]
    
    yb_full = label_binarize(yte, classes=range(n_cls))
    point   = average_precision_score(yb_full, prob, average='micro')
    
    boots = []
    for _ in range(n_boot):
        idx = rng.randint(0, len(yte), size=len(yte))
        yte_b  = yte[idx]
        prob_b = prob[idx]
        yb_b   = label_binarize(yte_b, classes=range(n_cls))
        # Skip resamples where any class is entirely absent
        if yb_b.sum(axis=0).min() == 0:
            continue
        try:
            boots.append(average_precision_score(yb_b, prob_b, average='micro'))
        except Exception:
            pass
    
    boots = np.array(boots)
    lo, hi = np.percentile(boots, [2.5, 97.5])
    return point, lo, hi, boots

# ── Compute for all groups ─────────────────────────────────────────────────────
groups_plot = [
    ('Norway',          norway_models),
    ('Norway Purified', norway_purified_models),
    ('TCGA',            tcga_models),
    ('TCGA Purified',   tcga_purified_models),
]

all_boot_rows = []

for group_name, entries in groups_plot:
    print(f'\n{"="*60}')
    print(f'Bootstrap CI — {group_name}  (n_boot={N_BOOT})')
    print(f'{"="*60}')
    
    boot_rows = []
    for lbl, r in entries:
        n_test = len(r['yte'])
        point, lo, hi, boots = bootstrap_micro_ap(r, n_boot=N_BOOT)
        ci_width = hi - lo
        print(f'  {lbl:<30s}  AP={point:.3f}  95% CI=[{lo:.3f}, {hi:.3f}]'
              f'  width={ci_width:.3f}  n_test={n_test}  n_boot_valid={len(boots)}')
        boot_rows.append(dict(group=group_name, model=lbl,
                              ap=point, ci_lo=lo, ci_hi=hi,
                              ci_width=ci_width, n_test=n_test))
        all_boot_rows.append(boot_rows[-1])
    
    if not boot_rows:
        continue
    
    # ── Error-bar plot for this group ──────────────────────────────────────────
    labels = [x['model'] for x in boot_rows]
    points = [x['ap']    for x in boot_rows]
    los    = [x['ci_lo'] for x in boot_rows]
    his    = [x['ci_hi'] for x in boot_rows]
    yerr   = np.array([[p - l, h - p] for p, l, h in zip(points, los, his)]).T

    fig, ax = plt.subplots(figsize=(10, 4.5))
    x_pos = np.arange(len(labels))
    
    # Color bars by whether CI crosses IRUS-RS (weakest model) — just use uniform color
    bar_colors = ['#e74c3c' if p < 0.55 else '#3498db' for p in points]
    ax.bar(x_pos, points, color=bar_colors, alpha=0.55, width=0.55,
           edgecolor='gray', linewidth=0.5, label='Micro-AP')
    ax.errorbar(x_pos, points, yerr=yerr, fmt='none',
                ecolor='black', capsize=5, capthick=1.8, elinewidth=1.5)
    
    ax.set_xticks(x_pos)
    ax.set_xticklabels(labels, rotation=40, ha='right', fontsize=9)
    ax.set_ylabel('Micro-Average PR-AUC')
    ax.set_title(f'{group_name} — Micro-AP with 95% Bootstrap CI  (n_boot={N_BOOT})',
                 fontsize=11)
    ax.set_ylim(0, 1.08)
    ax.axhline(y=0.5, color='gray', ls='--', lw=0.8, alpha=0.6, label='Chance (≈0.5)')
    ax.legend(fontsize=9)
    ax.grid(axis='y', alpha=0.25)
    plt.tight_layout()
    
    gname_safe = group_name.lower().replace(' ', '_')
    fname = f'{OUT}/bootstrap_ci_{gname_safe}.png'
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.show(); plt.close()
    print(f'  Saved {fname}')

# ── Save full CI table ─────────────────────────────────────────────────────────
boot_df = pd.DataFrame(all_boot_rows)
boot_df = boot_df.round({'ap': 4, 'ci_lo': 4, 'ci_hi': 4, 'ci_width': 4})
ci_csv = f'{OUT}/bootstrap_ci_table.csv'
boot_df.to_csv(ci_csv, index=False)
print(f'\nFull CI table saved → {ci_csv}')

# ── Combined plot: Norway vs Norway Purified side-by-side ─────────────────────
nw_rows = boot_df[boot_df['group'].str.startswith('Norway')]
if len(nw_rows) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
    for ax, gname in zip(axes, ['Norway', 'Norway Purified']):
        sub = nw_rows[nw_rows['group'] == gname]
        if sub.empty:
            continue
        labels = sub['model'].tolist()
        points = sub['ap'].tolist()
        los    = sub['ci_lo'].tolist()
        his    = sub['ci_hi'].tolist()
        yerr   = np.array([[p - l, h - p] for p, l, h in zip(points, los, his)]).T
        x_pos  = np.arange(len(labels))

        ax.bar(x_pos, points, color='#5dade2', alpha=0.6, width=0.55,
               edgecolor='gray', linewidth=0.5)
        ax.errorbar(x_pos, points, yerr=yerr, fmt='none',
                    ecolor='black', capsize=5, capthick=2, elinewidth=1.5)
        ax.set_xticks(x_pos)
        ax.set_xticklabels(labels, rotation=40, ha='right', fontsize=9)
        ax.set_title(f'{gname}\n(n_test={sub["n_test"].iloc[0]})', fontsize=11)
        ax.set_ylabel('Micro-Average PR-AUC')
        ax.set_ylim(0, 1.08)
        ax.axhline(y=0.5, color='gray', ls='--', lw=0.8, alpha=0.6)
        ax.grid(axis='y', alpha=0.25)

    plt.suptitle(f'Bootstrap 95% CI on Micro-AP — Norway cohorts  (n_boot={N_BOOT})',
                 fontsize=12)
    plt.tight_layout()
    fname = f'{OUT}/bootstrap_ci_norway_combined.png'
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.show(); plt.close()
    print(f'Saved {fname}')

print('\nBootstrap CI done.')
